<a href="https://colab.research.google.com/github/alessiomontagnerhermes-lgtm/Filosofia-computazionale-in-Colab/blob/main/Filosofia_computazionale_in_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Filosofia computazionale in Colab
**Introduzione all'uso di Vampire, Z3 e Lean per la filosofia**

##1. Introduzione

### 1.1 Perché la filosofia computazionale?

Prendiamo una qualsiasi proposizione creduta.

Di questa, è sempre possibile chiedere merito: "Perché dovremmo crederla?"

Renderne merito significa argomentarla.

Argomentarla significa, in ultimo luogo, derivarla logicamente come conclusione da altre proposizioni prese come premesse: se credo che "Socrate è mortale", lo argomento derivandolo da "Socrate è un uomo" e "ogni uomo è mortale".

Ma: l'argomentazione è valida? Davvero la conclusione è vera ogni volta che le premesse lo sono?

Possiamo rispondere formalizzando: rappresentiamo le proposizioni come formule logiche che trasformiamo secondo regole. Una formula ottenuta trasformandone altre è una loro conclusione: l'argomentazione è valida sotto quelle regole. Per esempio:

$\begin{array}{passo|formula|regola}
1. & U(S) & \text{Premessa} \\
2. & \forall x\,(U(x)\to M(x)) & \text{Premessa} \\
3. & U(S)\to M(S) & \forall E,\ 2 \\
4. & M(S) & \to E,\ 1,3
\end{array}$

dove

- $\forall E$ (*eliminazione dell'universale*) è la regola che, data una formula ($2$), la trasforma eliminando il quantificatore ($\forall$) e sostituendo le variabili ($x$) con termini individuali ($S$);

- $\to E$ (*eliminazione dell'implicazione*) è la regola che, data un'implicazione ($3$) e il suo antecedente ($1$), trasforma l'implicazione eliminando quest'ultimo.

Problema: chi assicura che le regole siano state applicate correttamente?

Ebbene, chi ha più autorità nel dire quanto fa 7414 + 3418: il proprio cervello, un professore di matematica, o una calcolatrice?

Statisticamente, la calcolatrice ha la minor probabilità d'errore: mentre gli esseri umani errano anche per fraintendimento, stanchezza, distrazione..., la calcolatrice erra solo in caso di bug software o errore hardware - due evenienze estremamente rare, essendo la calcolatrice molto semplice e continuamente controllata da professionisti.

Inoltre, mentre validare la risposta di un essere umano richiede di controllare ogni suo calcolo, la calcolatrice richiede di verificare solo il suo software: se questo è corretto, la correttezza di tutti i calcoli consegue. Ciò permette di verificare anche quei calcoli che sarebbero troppo lunghi e complessi per essere gestiti da un essere umano.

Tale esempio illustra le ragioni per le quali molti oggi considerano le metodologie computazionali le più rigorose.

Anche nel caso delle argomentazioni, la correttezza della derivazione di una conclusione dalle premesse può essere garantita applicando metodi computazionali.

In questo notebook vedremo come implementare alcuni metodi computazionali nella ricerca filosofica. Considereremo tre strumenti:

- Theorem provers

- Model finders

- Proof assistants

Vedremo come gestirli in *Colab* (colab.research.google.com). In primo luogo, Colab permette di creare e modificare notebook come questo tramite una semplice interfaccia grafica. In secondo luogo, mette a disposizione dell'utente potenza di calcolo e un ambiente Linux con preinstallati Python e alcune librerie, permettendogli di sfruttare queste risorse a prescindere dai mezzi di cui dispone localmente.

### 1.2 Theorem provers

Ogni argomentazione è riconducibile a due casi:

1. quelle che tentano di dimostrare che una certa proposizione ne contraddice un'altra;
2. e quelle che tentano di dimostrare che una certa proposizione *non* ne contraddice un'altra.

Le argomentazioni del primo tipo sono deduzioni. Infatti, dedurre una conclusione da delle premesse significa dimostrare che quelle premesse contraddicono la negazione della conclusione, cioè dimostrare che la negazione della conclusione è falsa ogni volta che le premesse sono vere.

I theorem prover automatici (ATP d'ora in poi) sono software ideali per le argomentazioni deduttive: dato un insieme di premesse e una conclusione, cercano una derivazione della conclusione dalle premesse. Se la trovano, si dimostra che l'argomentazione è valida.

L'ideale degli ATP, nato con il *calculus ratiocinator* di Leibniz, si è concretizzato per la prima volta nel 1956 con il *Logic Theorist* di Newell, Simon e Shaw. Molto influente è poi stato il *Prover9* di William McCune.

Noi utilizzeremo l'ATP *Vampire*, sviluppato da Voronkov, Hoder e Riazanov. Insieme ad *E* e *Spass*, è l'ATP oggi più utilizzato per la logica del primo ordine.

Ogni ATP affronta il seguente problema: come trovare una derivazione della formula-target dalle premesse in un tempo finito, quando da ogni premessa discendono infinite derivazioni di infinite formule? La tecnica adottata da *Vampire* è spiegata da Voronkov e Kovacs in queste lectures: https://vprover.github.io/vampireGuide/docs/lectures/l1/

Possiamo riassumere il procedimento come segue (senza approfondire i casi relativi all'uguaglianza, che hanno una gestione particolare):
1. Le formule delle premesse, più quella della negazione della conclusione, vengono riscritte in *forma normale skolemizzata*, cioè come congiunzioni di clausole. Le *clausole* sono disgiunzioni di letterali. I *letterali* sono atomi o loro negazioni. Gli *atomi* sono predicazioni su termini, oppure uguaglianze tra termini. I *termini* sono variabili, costanti, e simboli di funzione applicati a termini. Per esempio: $P$, $q$, $a$ e $b$ sono termini, $P(q)$ e $a=b$ sono letterali, $P(q) \lor a=b$ è una clausola.
2. I letterali vengono ordinati applicando l'algoritmo *Knuth-Bendix*, che assegna ad ogni loro termine un peso.
3. Si applica una *funzione di selezione* che seleziona in ogni clausola un letterale negativo oppure tutti i letterali massimali nell'ordinamento.
4. Si selezionano momentaneamente solo alcune clausole come *attive*, sulle quali si applicano i passaggi successivi.
5. Vengono escluse le clausole tautologiche (della forma $a \lor \lnot a \lor b$) e quelle sussunte (e.g., $a \lor b$ in presenza di $a$ o $b$).
6. Data una coppia di letterali uno positivo e uno negativo, si calcola il loro *most general unifier* (MGU), cioè una sostituzione di variabili tale da rendere quei letterali complementari (uguali tranne che per il segno di negazione), e si applica tale sostituzione alle clausole.
7. Le derivazioni sono calcolate solo sui letterali selezionati delle clausole attive e applicando il MGU.
8. Si cerca di derivare la clausola vuota (un $False$). Se ci si riesce, è dimostrato che dalle premesse si deriva la conclusione. Altrimenti, si rendono attive clausole prima lasciate passive e si riprova.

### 1.3 Model finders

Passiamo alle argomentazioni che tentano di dimostrare che le premesse *non* contraddicono un'altra proposizione. Ciò significa dimostrare che dalle premesse non si deriva la sua negazione.

Dimostrare che dalle premesse non si deriva $False$, e cioè che le premesse sono coerenti tra loro, è fondamentale. Infatti, per il principio *ex falso quodlibet* tutte le proposizioni possibili sono derivabili da un insieme di premesse contraddittorie. La derivazione della conclusione dimostrata dall'ATP sarebbe quindi valida ma vacua e non interessante, dato che avremmo potuto ugualmente derivare la sua negazione.

Inoltre, è utile dimostare che dalle premesse non si derivano tesi riteute indesiderabili: per esempio, se affermo di voler dimostrare l'esistenza di Dio senza usare il *Principio di Ragion Sufficiente* devo dimostrare che tale principio non si deriva dalle premesse adottate.

Un ATP, di norma, non può dimostrare questi casi di non-derivazione: anche se non riuscisse a trovare una derivazione, non significherebbe che questa non esista. Tali casi sono invece affrontati tramite teoria dei modelli.

Possiamo intendere una *estensione* come un insieme di fatti. Un *modello* è un'assegnazione di estensioni ai termini delle proposizioni tale da rendere tutte le proposizioni contemporaneamente vere: se una tale assegnazione è possibile, tali formule sono coerenti tra loro. Per esempio, prendiamo le formule

$U(S)$

$∀x(U(x)→M(x))$

e un insieme potenzialmente infinito di fatti. Possiamo creare il seguente modello:

$S=0$

$U=0$

$M=0$

In questo modello esiste un solo fatto, chiamato convenzionalmente $0$, identificato con $S$ e che è caratterizzato dalle proprietà $U$ ed $M$. Quindi $U(S)$ è immediatamente vero. E poiché tutti i fatti del modello che sono $U$ sono anche $M$, pure $∀x(U(x)→M(x))$ è vero.

I model finders funzionano in modo analogo agli ATP ma, anziché cercare derivazioni, cercano assegnazioni di estensioni ai termini.

Noi non utilizzeremo un model finder puro, ma un *Satisfiability Modulo Theories (SMT) solver*, cioè *Z3*, sviluppato da Leonardo de Moura per Microsoft Research. Il motivo della scelta è che Z3 è maggiormente versatile delle alternative: rispetto a *Mace 4* (il model finder di McCune), o anche a Vampire stesso in modalità `casc-sat` (cioè quando agisce come model finder), Z3 permette di gestire anche modelli con domini infiniti. Insieme a *CVC4* è il software più utilizzato nella sua categoria.

Quando usato come model finder, il funzionamento di Z3 può essere così descritto:

1. Z3 impiega *solver* specializzati in specifiche teorie, che gestiscono solo alcuni simboli. Uno stesso atomo può contenere simboli classifcati in teorie diverse. Poiché un atomo deve essere gestibile da un solver, si sostituiscono i termini con variabili finché ogni atomo non è riconducibile a un'unica teoria.
2. Le formule quindi vengono astratte sostituendo gli atomi logici con singole variabili proposizionali (e.g., $(a \rightarrow b) \lor (c \land d)$ diventa $v_1 \lor v_2$).
3. Per ogni variabile (quindi atomo) si cerca una assegnazione di valori di verità tale da rendere la formula complessivamente vera (e.g., $v_1 \lor v_2$ può essere intesa come $True \lor False$, oppure $False \lor True$, oppure $True \lor True$).
4. Tramite i solver, si controlla che tale assegnazione dei valori di verità rimanga possibile ricostruendo, dalle variabili proposizionali, gli atomi. Se non lo è, allora si aggiungono al sistema di formule delle *clausole di conflitto* che negano le assegnazioni di valori incompatibili (e.g., $\lnot a \lor b$ per rappresentare che se $a$ è vero $b$ non può essere falso). Quindi, si ripete la ricerca.
5. Una volta trovata un'assegnazione compatibile, si costruisce un modello concreto (cioè l'assegnazione di estensioni) per le formule in versione *ground*, cioè senza variabili e quindi senza quantificatori.
6. Trovato un modello per le formule ground, si prova ad estenderlo alla versione con quantificatori. Se non è possibile farlo, aggiunge ai vincoli da rispettare un controesempio a tale modello e se ne cerca un altro.

### 1.4 Proof assistants

I proof assistants sono strumenti diversi dai precedenti: mentre ATP e model finders producono automaticamente una dimostrazione o un modello, i proof assistants richiedono che sia l'utente a farlo, e si limitano a controllare la sua correttezza.

Perché usare un proof assistant se si hanno già ATP che fanno tutto da soli?

In primo luogo, le dimostrazioni prodotte dagli ATP sono spesso molto lunghe e oscure, richiedendo centinaia di passi anche per derivazioni banali. Un proof assistant permette di scrivere una derivazione breve e intelligibile.

In secondo luogo, un'argomentazione filosofica spesso richiede di usare sia un ATP che un model finder. Se i due software usano linguaggi diversi, come dimostrare che le traduzioni delle premesse nei due linguaggi rappresentano le stesse proposizioni? Bisognerebbe produrre un teorema di equivalenza in un metalinguaggio che formalizza al suo interno gli altri due linguaggi. Un proof assistant può svolgere entrambi i compiti tramite uno stesso linguaggio, senza richiedere traduzioni.

In terzo luogo, i proof assistant offrono un grado di certezza epistemica superiore agli altri software.

Infatti, i proof assistant garantiscono la correttezza delle prove tramite un *kernel*: un codice breve, relativamente semplice e continuamente controllato dai professionisti. Se il kernel è corretto, il proof assistant non può certificare come corretta una prova errata, esclusi casi estremi come errore hardware (un insetto in mezzo ai circuiti!) o un errore nel compilatore (il software che traduce il linguaggio usato dall'utente in linguaggio macchina).

Dopo il primo proof assistant, *Automath* di De Brujin, molti ne sono seguiti. Quello con il kernel più piccolo e affidabile è considerato *Metamath Zero*, ma il suo uso è molto difficoltoso.

Oggi è più usato *Lean*, sviluppato da Leonardo de Moura per Microsoft Research. Noi lo utilizzeremo perché permette di produrre prove molto più velocemente di Metamath Zero e, rispetto ad alternative come *Isabelle*, il suo *core* (la versione base del software) è sufficientemente leggero da essere utilizzato agevolmente in Colab.

Lean si basa sulla teoria dei tipi intuizionistica di Martin-Lof con tipi indipendenti. La *teoria dei tipi* è un sistema formale che assegna ad ogni oggetto in essa un tipo. Tale teoria è *intuizionistica* se non prevede il principio del terzo escluso ($P \lor \lnot P$). I *tipi indipendenti* sono oggetti che non dipendono dal valore assegnato come parametro (in opposizione a oggetti come il vettore $vector(n)$ che dipende dal valore di $n$).

In generale, il suo funzionamento può essere così descritto:
1. Lean accoglie delle dichiarazioni. Una *dichiarazione* può essere un oggetto come un assioma, una definizione, un teorema o altro. Ogni dichiarazione ha un *tipo* e, in alcuni casi, un *valore*.
2. Ogni volta che l'utente inserisce un termine, il kernel controlla se il suo tipo è definizionalmente uguale a quello atteso. Questo concetto di *definizionalmente uguale* dipende da una serie di algoritmi usati per tentare di ridurre due termini ad un medesimo termine. Per esempio, data una definizione `def a : T := v`, si controlla che il tipo di $v$ sia definizionalmente uguale al tipo $T$ assegnato ad $a$.
3. Un teorema viene "certificato" come corretto quando il kernel riscontra che la conclusione da dimostrare $C$ e un termine prodotto $t$ hanno tipi definizionalmente uguali.

## 2. Vampire

###2.1 Installazione

I file del software Vampire sono archiviati sul portale *GitHub*. Possiamo quindi scaricarlo tramite il software *wget*, che scarica file da un indirizzo web.

Per installare Vampire su un PC Linux apriremmo il terminale per avviare `wget` e gli daremmo come comando il link di download. In Colab vale la stessa procedura, ma prima di avviare wget bisogna dare comando al sistema di eseguire il codice come se lo stessimo scrivendo nel terminale, cosa che viene fatta con `!`. Inoltre, non volendo scaricare più volte lo stesso file in caso il codice venga ri-eseguito, daremo a `wget` il comando `-N`, che scarica i file dal link solo se sono più recenti di quelli installati. Quindi scriviamo:

```
!wget -N https://github.com/vprover/vampire/releases/download/v5.0.1/vampire-Linux-X64.zip
```

Questo scaricherà sempre la versione $5.0.1$ di Vampire. Questo assicura *riproducibilità*: chiunque ricontrollerà questo notebook eseguirà i codici usando sempre le stesse versioni dei software.

I file vengono scaricati dal link in una cartella compressa chiamata `vampire-Linux-X64.zip`. Per usare i file in essa contenuti dobbiamo prima decomprimerla dandola come comando al software `unzip`. Lo facciamo aggiungendo il comando `-u` che evita di decomprimere nuovamente e inutilmente la cartella in caso di ri-esecuzione del codice:

```
!unzip -u vampire-Linux-X64.zip
```

Il file che permetterà di eseguire Vampire è chiamato `vampire`. Per usarlo dobbiamo prima dare il permesso di eseguirlo. Ciò viene fatto tramite il software `chmod`, che prende come primo comando un permesso (quello di esecuzione si indica con `+x`) e come secondo il file a cui applicarlo. Quindi scriviamo:

```
!chmod +x vampire
```

Nella prossima cella possiamo vedere ed eseguire il codice completo.

In [ ]:
!wget -N https://github.com/vprover/vampire/releases/download/v5.0.1/vampire-Linux-X64.zip
!unzip -u vampire-Linux-X64.zip
!chmod +x vampire

--2026-10-15 15:20:49--  https://github.com/vprover/vampire/releases/download/v5.0.1/vampire-Linux-X64.zip
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/18067910/f28c77ef-07c2-4e6a-a5ce-6ad0620e3ed3?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-10-15T16%3A09%3A31Z&rscd=attachment%3B+filename%3Dvampire-Linux-X64.zip&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-10-15T15%3A09%3A11Z&ske=2026-10-15T16%3A09%3A31Z&sks=b&skv=2018-11-09&sig=h7uQGdtClmBu1ZdVNQ5t1fYobP%2FudA9cgODv%2BzQ8kQw%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc5MjA3OTQ0OSwibmJmIjoxNzkyMDc3NjQ5LCJwYXRoIjoicmVsZWFzZWFzc2V0cHJ

###2.2 Wrapper

Utilizzare software come Vampire richiede spesso di compiere diverse operazioni. Per facilitarne l'uso è consigliato creare una funzione Python che compia queste operazioni in automatico; cioè, è consigliato creare un *wrapper*.

Iniziamo definendo (`def`) questa funzione, che chiamiamo `prove_vampire`. Includerà due argomenti: la lista delle premesse `axioms` e la conclusione da derivare `goal`.

```
def prove_vampire(axioms, goal):
```

Utilizzare Vampire richiede di scrivere un file con estensione `.p` che includa il codice da eseguire. Creiamo quindi tale file con la funzione `open`: specifichiamo come suo primo argomento il nome del file (`theorem.p`), e come secondo argomento la modalità `w` che specifica di creare quel file se non esiste ancora:

```
def prove_vampire(axioms, goal):
  open("theorem.p", "w")
```

Quindi, andiamo a usare su tale file (a cui assegniamo l'alias `f` per brevità) la funzione `write` per scrivere in esso il codice, cioè le premesse `axioms` e la conclusione `goal`. Poniamo il tutto in `with`, che chiude in automatico il file alla fine della riga:

```
def prove_vampire(axioms, goal):
  with open("theorem.p", "w") as f: f.write(axioms + "\n" + goal)
```

Ora, per avviare Vampire dobbiamo utilizzare la funzione `run` del modulo `subprocess` (da importare con `import` all'inizio del codice). Tale funzione accetta più argomenti: il software da avviare, più varie impostazioni.

Come primo argomento, inseriamo una lista (`[...]`) di stringhe, che includerà il percorso del software da avviare più vari comandi per quel software. Poiché `vampire` si trova nella cartella corrente (chiamata per convenzione `.`) sarà sufficiente indicare come percorso `./vampire`:

```
subprocess.run(["./vampire"])
```

Come comandi da inviare a Vampire, indichiamo che l'esecuzione del programma deve avvenire in modalità di ricerca di derivazioni (`"--proof", "on"`, non come model finder) sul file `theorem.p`:

```
subprocess.run(["./vampire", "--proof", "on", "theorem.p"])
```

Come secondo e terzo argomento per `run` specifichiamo che l'output di Vampire verrà catturato come testo (`capture_output=True, text=True`): potremo decidere in seguito se stamparlo oppure no.

```
subprocess.run(["./vampire", "--proof", "theorem.p"], capture_output=True, text=True)
```

Passiamo quindi alla gestione della stampa dell'output. Quando Vampire trova una derivazione, stampa per convenzione l'espressione `SZS status Theorem` nel suo output (il `stdout` prodotto da `subprocess.run`). Possiamo porre che, se si trova tale espressione, si stampi una conferma:

```
if "SZS status Theorem" in process.stdout:
  print("✅ Proved")
```

`stdout` include anche la derivzione completa della conclusione dalle premesse. Affinché l'utente possa scegliere se stampare o meno tale output, inseriamo una variabile `proof_mode`, tale che l'utente possa cambiarne agilmente il valore. L'output sarà stampato (sotto la riga introduttiva introduttiva `---Proof:`) solo se `proof_mode` è stato impostato su `on`:

```
proof_mode = "on"

if "SZS status Theorem" in process.stdout:
  print("✅ Proved")
  if proof_mode == "on":
    print("\n--- Proof:")
    print(process.stdout)
```

In alcuni casi Vampire non deriva la conclusione perché è riuscito a dimostrare che essa non è derivabile. In tal caso l'output conterrà l'espressione `SZS status CounterSatisfiable`. Dunque, segnaliamo il risultato:

```
if "SZS status Theorem" in process.stdout:
  print("✅ Proved")
  if proof_mode == "on":
    print("\n--- Proof:")
    print(process.stdout)
elif "SZS Status CounterSatisfiable" in process.stdout:
  print("❌ Not proved")
  print("\n--- Output ---")
  print(process.stdout)
```

Infine, Vampire potrebbe non riuscire né a trovare la derivazione, né a dimostrare la sua impossibilità. Quando accade, cionondimeno `stdout` e, in alcuni casi, `stderr` possono contenere informazioni utili sui motivi della fallita ricerca o su errori nel codice. Stampiamo questi quando in tutti i casi diversi dai precedenti:

```
if "SZS status Theorem" in process.stdout:
  print("✅ Proved")
  if proof_mode == "on":
    print("\n--- Proof:")
    print(process.stdout)
elif "SZS Status CounterSatisfiable" in process.stdout:
  print("❌ Not proved")
  print("\n--- Output ---")
  print(process.stdout)
else:
  print("❔ Unknown")
  print("\n--- Output ---")
  print(process.stdout)
  if process.stderr:
    print("\n--- Errors ---")
    print(process.stderr)
```

Fuori dalla funzione, stampiamo una conferma del fatto che il codice è stata eseguito fino in fondo e quindi che il wrapper è stato salvato nel contesto:

```
print("✅ The wrapper has been loaded")
```

Di seguito è possibile vedere il codice completo e caricare il wrapper.

In [ ]:
import subprocess

def prove_vampire(axioms, goal):

  with open("theorem.p", "w") as f: f.write(axioms + "\n" + goal)

  process = subprocess.run(["./vampire", "--proof", "on", "theorem.p"], capture_output=True, text=True)

  proof_mode = "on"

  if "SZS status Theorem" in process.stdout:
    print("✅ Proved")
    if proof_mode == "on":
      print("\n--- Proof:")
      print(process.stdout)
  elif "SZS Status CounterSatisfiable" in process.stdout:
    print("❌ Not proved")
    print("\n--- Output ---")
    print(process.stdout)
  else:
    print("❔ Unknown")
    print("\n--- Output ---")
    print(process.stdout)
    if process.stderr:
      print("\n--- Errors ---")
      print(process.stderr)

print("✅ The wrapper has been loaded")

✅ The wrapper has been loaded


###2.3 Trovare una derivazione con Vampire

####2.3.1 Scrivere formule in *fof*

Vampire rappresenta le formule logiche secondo la sintassi *TPTP*. Questa ha numerose varianti. Noi ne vedremo due: *fof* per la logica del primo ordine, e *tff* per la logica del primo ordine *multisorta* (cioè con domini distinti per variabili di tipo diverso). Per approfondire la sintassi TPTP, si veda: https://tptp.org/UserDocs/TPTPLanguage/TPTPLanguage.shtml

Iniziamo traducendo in *fof* le formule

$U(S)$

$\forall x\,(U(x)\to M(x))$

$M(S)$

In *fof* le premesse e la conclusione da derivare vanno dichiarate in modo diverso. Le premesse si dichiarano come segue:

$\color{red}{\text{fof(}} \color{#56B4E9}{\text{nome_premessa}} \color{red}{\text{, axiom, }} \color{#56B4E9}{\text{formula_premessa}} \color{red}{).}$

La conclusione invece va dichiarata come segue:

$\color{red}{\text{fof(}} \color{#56B4E9}{\text{nome_conclusione}} \color{red}{\text{, conjecture, }} \color{#56B4E9}{\text{formula_conclusione}} \color{red}{).}$

dove le parti in rosso sono fisse, mentre quelle azzurre sono variabili definite dall'utente.

Contrariamente alla prassi standard in logica matematica, in TPTP
- le costanti e i predicati sono indicate con lettere minuscole;
- le variabili sono indicate con lettere maiuscole.

Inoltre:
- il quantificatore universale è indicato con `![x] :`, avente tra parentesi la lista delle variabili quantificate;
- l'implicazione materiale è indicata con `=>`.

Possiamo quindi tradurre le formule come segue, chiamando le premesse `a1` e `a2` rispettivamente e la conclusione `c`:

```
fof(a1, axiom, u(s)).
fof(a2, axiom, ![X] : (u(X) => m(X))).
fof(c, conjecture, m(s)).
```

Al fine di verificare se la conclusione effettivamente derivi dalle premesse, assegniamo le premesse alla variabile `axioms` e la conclusione alla variabile `conclusion`, e indichiamole poi come argomenti del wrapper `prove_vampire`. Eseguendo la cella, viene prodotta la derivazione.

In [ ]:
axioms = r"""
fof(a1, axiom, u(s)).
fof(a2, axiom, ![X] : (u(X) => m(X))).
"""

conclusion = r"""
fof(c, conjecture, m(s)).
"""

prove_vampire(axioms, conclusion)

✅ Proved

--- Proof:
% Running in auto input_syntax mode. Trying TPTP
% Refutation found. Thanks to Tanya!
% SZS status Theorem for theorem
% SZS output start Proof for theorem
1. u(s) [input(axiom)]
2. ! [X0] : (u(X0) => m(X0)) [input(axiom)]
3. m(s) [input(conjecture)]
4. ~m(s) [negated conjecture 3]
5. ~m(s) [flattening 4]
6. ! [X0] : (m(X0) | ~u(X0)) [ennf transformation 2]
7. u(s) [cnf transformation 1]
8. ~u(X0) | m(X0) [cnf transformation 6]
9. ~m(s) [cnf transformation 5]
10. m(s) [resolution 8,7]
11. $false [forward subsumption resolution 10,9]
% SZS output end Proof for theorem
% ------------------------------
% Version: Vampire 5.0.1 (Release build, commit 1b13eaf on 2026-01-18 12:14:50 +0000)
% Linked with Z3 4.14.0.0 3c47fd96cf5645d0c42b2c819d9e9a84380aa721 NOTFOUND
% CaDiCaL version: 2.1.3
% Termination reason: Refutation
% Time elapsed: 0.001 s
% Peak memory usage: 159 MB
% ------------------------------
% ------------------------------



###2.3.2 Scrivere formule in *tff*

*tff* permette di assegnare tipi distinti alle variabili, potendo quindi rappresentare una logica del primo ordine multisorta. Poiché sia la logica modale, sia la logica del secondo ordine con semantica di Henkin, sono traducibili in logica del primo ordine multisorta, è presumibile che *tff* sarà sufficiente a gran parte degli scopi del filosofo.

Proviamo a rappresentare in *tff* le premesse di un semplice argomento ontologico:

$\forall w \forall v A(w,v)$

$\exists w In(g,w)$

$\forall w \forall v ((In(g,w) \land A(w,v)) \rightarrow In(g,v))$

da cui si vuole derivare la conclusione

$In(g,w_a)$

Rispetto a *fof*, *tff* richiede di definire separatamente le *sorti* (i domini delle variabili) e le *segnature* di costanti e predicati (cioè i domini d'appartenenza dei loro argomenti).

I domini vengono dichiarati secondo questo schema:

$\color{red}{\text{tff(}} \color{#56B4E9}{\text{nome_formula}} \color{red}{\text{, type, }} \color{#56B4E9}{\text{nome_dominio}} \color{red}{: \ $tType).}$

A noi serve un dominio per le entità (come $g$) e un altro per i mondi (come $w$), quindi scriveremo:

```
tff(entity_type, type, entity: $tType).
tff(world_type, type, world: $tType).
```

Le costanti, invece sono dichiarate secondo lo schema

$\color{red}{\text{tff(}} \color{#56B4E9}{\text{nome_formula}} \color{red}{\text{, type, }} \color{#56B4E9}{\text{nome_costante}} \color{red}{\text {: }} \color{orange}{\text {nome_dominio}}  \color{red}{\text {).}}$

dove il testo arancione sta a indicare che dobbiamo scrivere l'esatto nome del dominio che appare nella sua dichiarazione. Noi andremo a dichiarare due costanti, cioè la costante $g$, che è un'entità, e la costante $w_a$, che è un mondo:

```
tff(god_type, type, g: entity).
tff(actual_type, type, w_a: world).
```

Infine, i predicati vanno dichiarati secondo questo schema:

$\color{red}{\text{tff(}} \color{#56B4E9}{\text{nome_formula}} \color{red}{\text{, type, }} \color{#56B4E9}{\text{nome_predicato}} \color{red}{\text {: (}} \color{orange}{\text {nome_dominio_primo_argomento}} \color{red}{\text { * }} \color{orange}{\text {nome_dominio_secondo_argomento}} \color{red}{) > $o).}$

Ovviamente, in caso di predicati con un numero di argomenti diverso da 2, lo schema varia di conseguenza. Nel nostro caso abbiamo il predicato $In$, che accetta come argomenti un'entità e un mondo, e il predicato $A$, che ha come argomenti due mondi:

```
tff(exists_type, type, in: entity > world > $o).
tff(accessible_type, type, a: world > world > $o).
```

La struttura delle formule di premesse e conclusioni, invece, è la medesima di *fof*, con due differenze:
- ovviamente, ogni formula iniza con `tff(` anziché con `fof(`;
- le variabili dei quantificatori devono specificare il loro tipo, secondo lo schema `![x: nome_dominio]` per la quantificazione universale e `?[x: nome_dominio]` per la quantificazione esistenziale.

Traduciamo quindi premesse e conclusioni come segue:

```
tff(a1, axiom, ![W:world,V:world] : a(W,V)).
tff(a2, axiom, ?[W:world] : in(g,W)).
tff(a3, axiom, ![W:world,V:world] : ((in(g,W) & a(W,V)) => in(g,V))).
tff(c, conjecture, in(g,w_a))
```

Per testare la derivazione, usiamo nuovamente il wrapper `prove_vampire`. Sorti, costanti e predicati possono essere inseriti direttamente nella medesima variabile delle premesse.

In [ ]:
axioms = r"""
% SORTS
tff(entity_type, type, entity: $tType).
tff(world_type, type, world: $tType).

%CONSTANTS
tff(god_type, type, g: entity).
tff(actual_type, type, w_a: world).

%PREDICATES
tff(exists_type, type, in: (entity * world) > $o).
tff(accessible_type, type, a: (world * world) > $o).

%AXIOMS
tff(a1, axiom, ![W:world,V:world] : a(W,V)).
tff(a2, axiom, ?[W:world] : in(g,W)).
tff(a3, axiom, ![W:world,V:world] : ((in(g,W) & a(W,V)) => in(g,V))).
"""

conclusion = r"""
tff(c, conjecture, in(g,w_a)).
"""

prove_vampire(axioms, conclusion)

✅ Proved

--- Proof:
% Running in auto input_syntax mode. Trying TPTP
% Refutation found. Thanks to Tanya!
% SZS status Theorem for theorem
% SZS output start Proof for theorem
1. ! [X0 : world,X1 : world] : a(X0,X1) [input(axiom)]
2. ? [X0 : world] : in(g,X0) [input(axiom)]
3. ! [X0 : world,X1 : world] : ((in(g,X0) & a(X0,X1)) => in(g,X1)) [input(axiom)]
4. in(g,w_a) [input(conjecture)]
5. ~in(g,w_a) [negated conjecture 4]
6. ~in(g,w_a) [flattening 5]
7. ! [X0 : world,X1 : world] : (in(g,X1) | (~in(g,X0) | ~a(X0,X1))) [ennf transformation 3]
8. ! [X0 : world,X1 : world] : (in(g,X1) | ~in(g,X0) | ~a(X0,X1)) [flattening 7]
9. ? [X0 : world] : in(g,X0) => in(g,sK0) [skolem symbol introduction]
10. in(g,sK0) [skolemisation 2,9]
11. a(X0,X1) [cnf transformation 1]
12. in(g,sK0) [cnf transformation 10]
13. ~a(X0,X1) | ~in(g,X0) | in(g,X1) [cnf transformation 8]
14. ~in(g,w_a) [cnf transformation 6]
15. ~in(g,X0) | in(g,X1) [resolution 13,11]
17. 1 <=> ! [X1 : world] : in(g,X1) [avatar defin

Nel caso si voglia testare se una certa conclusione deriva dalle medesime premesse `axioms`, ma la nuova `conclusion` richieda di dichiarare predicati o altri segni non già inclusi in `axioms`, si possono aggiungere tali dichiarazioni sopra la nuova conclusione nella medesima variabile: il wrapper `prove_vampire` si limiterà a unire i blocchi di testo di `axioms` e `conclusion` in un medesimo file.

Similmente, se si vogliono testare estensioni del sistema di premesse attualmente in uso, è sufficiente definire una nuova variabile e aggiungere le nuove premesse secondo lo schema $\color{#56B4E9}{\text{nome_variabile}} \color{red}{\text{ = axioms + """}} \color{#56B4E9}{\text{nuove_premesse}} \color{red}{\text {"""}}$.

## 3. Z3

###3.1 Installazione

Z3 può essere installato tramite `pip`, un software che ricerca (e, dandogli come primo comando `install`, scarica e installa) pacchetti di file in server dedicati.

Il pacchetto contenente Z3 è chiamato `z3-solver`, e va dato come secondo comando a `pip`. Eseguiremo quindi il seguente codice:

```
!pip install z3-solver
```

Questo, però, installa l'ultima versione disponibile di Z3. Come assicurare la riproducibilità?

Una volta installato Z3, la cella mostrerà un output contenente il numero della versione secondo la forma `Successfully installed z3-solver-NUMERO_VERSIONE`

Quindi, si modifica il codice aggiungendo alla fine `==NUMERO_VERSIONE`

Con la cella seguente installeremo la versione 5.0.0.0.

In [ ]:
!pip install z3-solver==5.0.0.0

###3.2 Wrapper

Il `z3-solver` che abbiamo appena scaricato contiene la *API* Python per Z3: cioè, permette di comunicare con Z3 tramite il linguaggio Python, anziché dover creare un file in linguaggio *SMT-LIB* da eseguire con `subprocess`. Si tratta di una implementazione ufficiale per Z3 che non ha equivalenti per Vampire e Lean (se non tramite software di terze parti spesso non aggiornati). Questo facilita in generale l'uso di Z3, e in particolare la stampa dei modelli (che altrimenti richiederebbe di "tagliuzzare" l'output tramite *parsing*). Il wrapper dovrà differenziarsi da quello di Vampire di conseguenza.

Iniziamo definendo una funzione `prove_z3`. Z3 richiede di definire i *tipi* (domini) e alcuni simboli (variabili, costanti, predicati) prima di poter rappresentare le premesse. Indichiamo quindi tre argomenti per la funzione: un `system` che include domini e simboli, `axioms` che include le premesse, e `test_formula` che indica la formula per la quale vogliamo testare la coerenza con le premesse. Poiché non è necessario indicare una `test_formula` se si vuole testare la coerenza delle sole premesse tra loro, poniamo il terzo argomento come `True` di default (cioè come una formula che non può modificare la coerenza delle premesse):

```
def prove_z3(system, axioms, test_formula="True"):
```

`system` e `axioms` dichiarano domini, simboli e premesse tramite variabili e dizionari Python. `system` e `axioms`, però, saranno a loro volta racchiusi in stringhe. Per questo, affinché ciò che scriviamo in essi entri nel sistema, dobbiamo eseguirli con la funzione `exec`:

```
exec(system)
exec(axioms)
```

Z3 ricerca modelli tramite dei solver formalizzati in Python come delle classi. Tali classi hanno i loro metodi (funzioni), tra i quali `add`, che viene usato per aggiungere al solver le formule da testare.

Possiamo quindi creare un oggetto `Solver`, assegnandolo ad una variabile `solver`, e poi usare il metodo `add` per aggiungere le premesse espresse nel dizionario `axioms` e la `test_formula`. Questa `test_formula` verrà espressa tramite una stringa usando direttamente i simboli dichiarati in `system`, e a differenza di `axioms` non è un normale oggetto Python. Dobbiamo quindi applicare la funzione `eval`, che la "riscrive" sulla base delle dichiarazioni in `system` e possa quindi essere utilizzata da Z3:

```
solver = Solver()

for name, axiom in axioms.items():
  solver.add(axiom)

solver.add(eval(test_formula))
```

Benché questo codice possa funzionare, potrebbe farlo una volta sola. Il motivo è che `system` viene ri-caricato da `exec` ad ogni esecuzione della funzione. Ma `system` potrebbe contenere domini che Z3 non permette di dichiarare più volte, di modo che ri-eseguire `exec(system)` porterebbe ad un un errore nel sistema. Come evitarlo?

Z3 permette di relativizzare i domini ad un contesto, rappresentato tramite la classe `Context`. Iniziamo creando una sua istanza e assegnandola alla variabile `context`:

```
context = Context()
```

I domini di `system` saranno relativizzati a tale contesto: ogni chiamata crea un nuovo contesto e quindi nuovi tipi, invece di ri-dichiararli.

Ora, quando `exec(system)` costruisce `system` lo fa andando a prendere gli elementi necessari da un dizionario. Alcuni di questi elementi si trovano nel contesto globale `globals`. Affinché i domini del `system` siano relativizzati al contesto, però, `exec` deve essere in grado di riferirsi a `context`, che si trova invece nel contesto locale. Come rimediare?

L'idea è quella di usare la funzione `copy` per creare una copia di `globals` e assegnarla alla variabile `namespace`; quindi, andremo ad aggiungere `context` in esso e indicheremo `namespace` come secondo argomento di `exec`, cioè come il dizionario che userà nel costruire `system`:

```
namespace = globals().copy()
namespace["context"] = context

exex(system, namespace)
exec(axioms, namespace)
```

Dobbiamo fare la stessa operazione anche per la `text_formula` in `eval`. Inoltre, anche il solver viene definito come appartenente al contesto corrente, che viene assegnato al suo argomento `cxt`:

```
solver = Solver(ctx=context)

solver.add(eval(test_formula, namespace))
```

Questo è sufficiente ad assicurare che non ci saranno conflitti di nomi anche con chiamate multiple della funzione.

Un'ultima cosa da fare è inserire la possibilità di escludere una delle premesse dal solver. Infatti, come vedremo, alcuni test richiedono di cercare modelli escludendo una premessa alla volta.

Per farlo, dobbiamo anzituto aggiungere a `prove_z3` un argomento, `exclude`, che specifichi l'assioma escluso, e poi un argomento `exclude_loop` che permetta all'utente di scegliere se eseguire o meno questo tipo di test. Questi argomenti vengono impostati, di default, su `None` e `False` rispettivamente, di modo che vadano specificati solo quando utili:

```
def prove_z3(system, axioms, test_formula=None, exclude=None, exclude_loop=False):
```

Dobbiamo quindi creare un *for loop* che, per ogni premessa in `axioms` (che ora è una voce di `namespace`) esegua `prove_z3` dando ad `exclude` il valore della premessa sotto esame:

```
for target_axiom in namespace['axioms']:
    prove_z3(system, axioms, test_formula=test_formula, exclude=target_axiom)
```

Per chiarezza, stampiamo nell'output, per ogni interazione, un messaggio che ci dica quale premessa sta venendo esclusa, seguita dal risultato. Poniamo il futto sotto la condizione che `exclude_loop` sia stato impostato come `True`. Inoltre, dobbiamo limitare l'aggiunta degli `axioms` al solver alle sole premesse il cui nome non è in `exclude`:

```
if exclude_loop:
    for target_axiom in namespace['axioms']:
         print(f"Testing without ({target_axiom}):")
         prove_z3(system, axioms, test_formula=test_formula, exclude=target_axiom)
    return

for name, axiom in namespace['axioms'].items():
    if name != exclude: solver.add(axiom)
```

La gestione della stampa dell'output è analoga a quella già vista in Vampire.

Nella cella seguente possiamo vedere il codice completo ed eseguirlo.

In [ ]:
def prove_z3(system, axioms, test_formula="True", exclude=None, exclude_loop=False):

    context = Context()

    namespace = globals().copy()
    namespace["context"] = context

    exec(system, namespace)
    exec(axioms, namespace)

    if exclude_loop:
        for target_axiom in namespace['axioms']:
            print(f"\nTesting without ({target_axiom}):")
            prove_z3(system, axioms, test_formula=test_formula, exclude=target_axiom)
        return

    solver = Solver(ctx=context)

    for name, axiom in namespace['axioms'].items():
        if name != exclude: solver.add(axiom)

    solver.add(eval(test_formula, namespace))

    result = solver.check()
    show_models = "on"

    if result == sat:
        print("✅ Sat")
        if show_models == "on":
            print("\n--- Model ---")
            print(solver.model())
    elif result == unsat:
        print("❌ Unsat")
    else:
        print("❓ Unknown")
        print("\n--- Reason ---")
        print(solver.reason_unknown())

print("✅ The wrapper has been loaded")

✅ The wrapper has been loaded


###3.3 Testare proprietà logiche con Z3

####3.3.1 Test di coerenza

In modo simile a quanto visto con *tff*, anche il linguaggio di Z3 richiede di dichiarare esplicitamente domini, predicati, costanti, e anche le variabili. Ciò viene fatto tramite specifiche funzioni Python.

Anzitutto, prima del primo uso del wrapper bisogna impartire il comando `from z3 import *`, che importa tutti i *nomi pubblici* di Z3, incluse quindi le funzioni che andremo ad usare per le dichiarazioni.

Per dichiarare un dominio si usa la funzione `DeclareSort`, a cui andremo a indicare come argomenti il nome che diamo a tale dominio e il contesto (tramite l'argomento opzionale `ctx`). Non sarà necessario relativizzare al contesto variabili, costanti e funzioni in quanto saranno già relativizzate in virtù della loro appartenenza a un certo dominio. Riprendendo l'esempio dell'argomento ontologico, dichiariamo due domini, e li assegniamo alle variabili `E_Type` (per le entità) e `W_Type` (per i mondi):

```
E_Type = DeclareSort("Entity", ctx=context)
W_Type = DeclareSort("World", ctx=context)
```

Si noti che in Z3 è necessario dichiarare il dominio anche nel caso di logica del primo ordine non-multisorta. Inoltre, esistono modi alternativi per dichiarare i domini. Per esempio, nel caso di domini finiti, è possibile utilizzare la funzione `EnumSort`, che accetta come secondo argomento la lista degli specifici enti appartenenti a quel dominio. Così, se avessimo voluto rendere `E_Type` una `EnumSort` di cardinalità $3$ avremmo scritto:

```
E_Type, (a, b, c) = EnumSort("Entity", ["a", "b", "c"], ctx=context)
```

In modo simile, le costanti e le variabili vanno dichiarate con la funzione `Const`, che accetta come argomenti un nome e il dominio d'appartenenza. Benché possa sembrare controintuitivo dichiarare una variabile con `Const`, Z3 distinguerà costanti e variabili a seconda del loro essere quantificate o meno nelle formule:

```
a = Const("a", E_Type)
w = Const("w", W_Type)
v = Const("v", W_Type)
w_a = Const("w_a", W_Type)
g   = Const("g", E_Type)
```

Infine, i predicati vanno dichiarati con la funzione `Function`, interpretandoli quindi come funzioni che restituiscono un $booleano$ (valore di verità). `Function` accetta come argomenti il nome del predicato, i nomi dei domini degli argomenti del predicato, e infine la funzione dei booleani `BoolSort`, per la quale specifichiamo il suo argomento `ctx` come fatto con `DeclareSort` (onde evitare un *mismatch* di contesti):

```
In = Function("In", E_Type, W_Type, BoolSort(ctx=context))
A  = Function("A", W_Type, W_type, BoolSort(ctx=context))
```

Per tradurre le premesse nel linguaggio di Z3 dobbiamo considerare che tutti gli operatori logici sono trattati come funzioni le cui proposizioni vanno indicate come loro argomenti, analogalmente ai predicati. Per esempio, la formula

$a \land b$

va riscritta secondo la logica

$\land(a,b)$

Questo crea una struttura annidata "ad albero". Per esempio, se abbiamo

$(a \lor b) \land (c \rightarrow d)$

la sua struttura sarà

$\land(\lor(a,b),\rightarrow(c,d))$

In Z3, $\land$ si indica con `And`, $\rightarrow$ con `Implies`, $\forall$ con `ForAll`, $\exists$ con `Exists`.

Al fine di poterci riferire alle premesse "per nome" e facilitare i *loop* per i test che escludono una premessa alla volta, le inseriamo in un dizionario:

```
"axiom1": ForAll([w,v], A(w,v)),
"axiom2": Exists([w], In(g,w)),
"axiom3": ForAll([w,v], Implies(And(In(g,w), A(w,v)), In(g,v)))
```

Per poter eseguire il wrapper `prove_z3`, assegniamo le dichiarazioni di domini, costanti, variabili e predicati ad una variabile `system`, e il dizionario delle premesse alla variabile `axioms`. Benché la *API* di Z3 permetta di eseguire le dichiarazioni direttamente nel contesto globale (e assegnarle a variabili come stringhe ha lo svantaggio di richiedere `exec` ed `eval` nel nostro wrapper), questo consente di gestire più facilmente sistemi formali alternativi all'interno della stessa sessione di lavoro.

Anzitutto, vogliamo testare la coerenza delle premesse poste. Per farlo, è sufficiente assegnare il sistema logico e le premesse a `prove_z3` senza specificare un terzo argomento.

In [ ]:
from z3 import *

system = """
# --- Domain ---
E_Type = DeclareSort("Entity", ctx=context)
W_Type = DeclareSort("World", ctx=context)

# --- Variables ---
a = Const("a", E_Type)
w = Const("w", W_Type)
v = Const("v", W_Type)

# --- Constants ---
w_a = Const("w_a", W_Type)
g   = Const("g", E_Type)

# --- Predicates ---
In = Function("In", E_Type, W_Type, BoolSort(ctx=context))
A  = Function("A", W_Type, W_Type, BoolSort(ctx=context))
"""

axioms = """
axioms = {
  "axiom1": ForAll([w,v], A(w,v)),
  "axiom2": Exists([w], In(g,w)),
  "axiom3": ForAll([w,v], Implies(And(In(g,w), A(w,v)), In(g,v)))
}
"""

prove_z3(system, axioms)

✅ Sat

MODEL:
[g = Entity!val!0, In = [else -> True], A = [else -> True]]


####3.3.2 Test di indipendenza

Oltre alla coerenza delle premesse, è utile controllare che esse non implichino conseguenze indesiderabili.

Per esempio, vogliamo controllare che le nostre premesse non permettano di concludere `In(g,w_a)` solo perché in realtà da esse si deriva l'esistenza di ogni entità in ogni mondo. Per farlo, formaliziamo questa conseguenza indesiderata e la assegniamo come stringa ad una variabile `collapse`:

```
collapse = "ForAll([a,w], In(a,w))"
```

Questa formula non si deriva dalle nostre premesse se esse sono coerenti con la sua negazione. Possiamo quindi impostare come nostra `test_formula` una *stringa-f* della forma `Not({collapse})`.

In [ ]:
collapse = "ForAll([a,w], In(a,w))"

prove_z3(system, axioms, test_formula=f"Not({collapse})")

✅ Sat

MODEL:
[g = Entity!val!0,
 In = [else ->
       Not(And(Var(0) == Entity!val!1,
               Var(1) == World!val!1))],
 A = [else -> True]]


####3.3.3 Test di minimalità

Chiamiamo *minimalità* la proprietà di un sistema di premesse tale che
- l'eliminazione di una qualsiasi delle premesse non permette più di derivare la conclusione voluta;
- nessuna premessa è derivabile dall'insieme delle altre premesse.

La prima proprietà implica la seconda per la *regola del taglio*: se la conclusione è derivabile da un set di premesse $P$, e la premessa $x$ appartenente a $P$ è derivabile dalle altre premesse $P \setminus \{x\}$, allora la conclusione è derivabile dalle sole $P \setminus \{x\}$.

Quindi, per determinare la minimalità, è sufficiente dimostrare che, se si esclude una qualsiasi delle premesse, la conclusione non è più derivabile. Cioè, dimostrare che, escludendo una premessa alla volta, le rimanenti sono coerenti con la negazione della conclusione. Ciò può essere fatto automaticamente tramite l'opzione `exclude_loop` del wrapper `prove_z3`, ponendo la negazione della conclusione (`In(g,w_a)`) come `target_formula`.

Questo test non è essenziale come quello della coerenza: un sistema di premesse può voler includere affermazioni non essenziali alla derivazione della conclusione (per esempio, perché sta modellizzando un fenomeno generale da cui derivare diverse conclusioni). Ciò nonostante, un sistema di premesse minimale riduce il suo *impegno ontologico*, cioè le sue assunzioni, ed è quindi "più facilmente accettabile": non corre il rischio di essere rifiutato sulla base di una premessa non essenziale alla conclusione.

In [ ]:
conclusion = "In(g,w_a)"

prove_z3(system, axioms, test_formula=f"Not({conclusion})", exclude_loop=True)


Testing without (axiom1):
✅ Sat

--- Model ---
[g = Entity!val!0,
 w_a = World!val!1,
 In = [else ->
       Not(And(Var(0) == Entity!val!0,
               Not(Var(1) == World!val!0)))],
 A = [else -> False]]

Testing without (axiom2):
✅ Sat

--- Model ---
[g = Entity!val!0,
 w_a = World!val!0,
 In = [else -> False],
 A = [else -> True]]

Testing without (axiom3):
✅ Sat

--- Model ---
[g = Entity!val!0,
 w_a = World!val!1,
 In = [(Entity!val!0, World!val!1) -> False, else -> True],
 A = [else -> True]]


Z3 è un software estremamente versatile che permette di svolgere anche molti altri compiti. Per una guida, si veda: https://ericpony.github.io/z3py-tutorial/guide-examples.htm

## 4. Lean

###4.1 Installazione

####4.1.1 Installazione del core

Lean viene installato tramite il gestore *Elan*, un software che cerca e installa specifiche versioni di Lean. Il codice di Elan è archiviato su GitHub. A differenza di quanto fatto con Vampire, non abbiamo bisogno di salvare il file (stiamo scaricando il gestore, non Lean): è sufficiente eseguire immediatamente lo script Elan appena scaricato. Questo viene fatto usando `curl`.

Facciamo lavorare `curl` in modalità `-sSf`, una concatenazione di tre comandi che chiedono a `curl` di sopprimere i messaggi stampati se non quelli di errore:

```
!curl -sSf https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh
```

Come detto, dobbiamo inviare direttamente lo script di Elan alla *shell* (è come se lo eseguissimo nel terminale). Questo viene fatto combinando `curl` con `| sh -s`: la *pipe* `|` prende l'output di `curl` e lo invia come input al successivo `sh`, cioè la shell, con `-s` che specifica che i comandi arriveranno tramite input. Eseguendo tale script ci verrebbero inviate delle richieste, alle quali rispondiamo automaticamente "sì" con `-y`.

```
!curl -sSf https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh | sh -s -- -y
```

Come con Vampire, non vogliamo che il codice ri-scarichi Elan quando viene ri-eseguito. Non possiamo però gestire la cosa con un controllo di date (com'era stato fatto con `wget`), dato che non abbiamo salvato Elan su un file. Come fare?

Quando Elan viene scaricato, si crea un eseguibile chiamato `elan`. Possiamo quindi usare la funzione `which` del modulo `shutil` per controllase se questo eseguibile esiste oppure no, e scaricarlo solo se non c'è:

```
import shutil

if shutil.which("elan") is None:
  !curl -sSf https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh | sh -s -- -y
```

`which` cercherà `"elan"` nei percorsi indicati nel `PATH`. Dunque `which` potrebbe fallire se Elan è installato ma il suo percorso non è stato aggiunto al `PATH`. Dunque, prima del controllo di `which`, controlliamo che il percorso di Elan sia in `PATH`, e se non c'è, lo aggiungiamo manualmente. Questo richiede di importare il modulo `os`, essendo il `PATH` una voce del suo dizionario `environ`:

```
import os, shutil

if "/root/.elan/bin" not in os.environ["PATH"]:
    os.environ["PATH"] += ":/root/.elan/bin"
```

A questo punto, sorge di nuovo il problema della stabilità: come assicurare che chi ri-eseguirà questo codice a distanza di tempo usi sempre la stessa versione dei sofrware? Infatti, un aggiornamento di Lean potrebbe rendere invalida una dimostrazione oggi accettata. Anzitutto, controlliamo la versione installata di Lean con il seguente comando:

```
!lean --version
```

Quindi, inviamo allo script di Elan nella shell il comando `--default-toolchain` seguito dalla versione di Lean che vogliamo sia installata. Noi andremo a indicare la versione $4.34.1$:

```
import os, shutil

if "/root/.elan/bin" not in os.environ["PATH"]:
    os.environ["PATH"] += ":/root/.elan/bin"

if shutil.which("elan") is None:
  !curl -sSf https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh | sh -s -- -y --default-toolchain leanprover/lean4:v4.34.1
```

Ora vogliamo controllare che l'installazione sia andata a buon fine. Essendo articolata ed essendoci quindi molti punti in cui può fallire, il modo più sicuro per fare tale controllo è tentare di eseguire un file `.lean`.

Usiamo quindi la funzione `makedirs` di `os` per creare una cartella `/content/lean`, e specifichiamo come secondo argomento `exist_ok=True` affinché la funzione non generi un errore nel caso la cartella esistesse già. Usiamo quindi `open` e `write` per creare un file `test.lean` che conterrà solo il testo `eval 1+1`, un codice accettato da tutte le versioni di Lean:

```
os.makedirs("/content/lean", exist_ok=True)
with open("/content/lean/test.lean", "w") as f: f.write("#eval 1 + 1\n")
```

Quindi, usiamo la funzione `run` di `subprocess` dandogli come solo argomento la lista che chiede di eseguire `lean` sul file appena creato, più l'istruzione di catturare l'output come testo per poterlo stampare in caso di errore:

```
result = subprocess.run(["lean", "/content/lean/test.lean"], capture_output=True, text=True)
```

Quando il file `.lean` compila, Lean produce un attributo `returncode` a cui assegna valore $0$. Stamperemo quindi la conferma di corretta installazione quando `returncode == 0`. Altrimenti, stampiamo un messaggio di errore.

Di seguito, il codice completo per l'installazione di Lean.

In [ ]:
import os, shutil, subprocess

if "/root/.elan/bin" not in os.environ["PATH"]:
    os.environ["PATH"] += ":/root/.elan/bin"

if shutil.which("elan") is None:
    !curl -sSf https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh | sh -s -- -y --default-toolchain leanprover/lean4:v4.34.1

os.makedirs("/content/lean", exist_ok=True)
with open("/content/lean/test.lean", "w") as f: f.write("#eval 1 + 1\n")

result = subprocess.run(["lean", "/content/lean/test.lean"], capture_output=True, text=True)

if result.returncode == 0:
    print("✅ Lean has been installed successfully")
else:
    print("❌ Something went wrong, try again")
    print(result.stderr)

✅ Lean has been installed successfully


####4.1.2 Installazione di Mathlib

Il codice della sezione precedente installa solo gli elementi essenziali di Lean. In molti casi, è desiderabile installare anche *Mathlib*, una enorme raccolta di risultati matematici già dimostrati. Senza Mathlib, scrivere dimostrazioni relative a certi settori matematici richiederebbe di ricreare tali settori da zero.

Anzitutto, entriamo nella cartella `content` con il comando `%cd` e qui inviamo al programma `lake` il comando `new Project` per creare un nuovo progetto, cioè una serie di cartelle preimpostate per i file Lean. Indichiamo di crearlo secondo il modello `math`, cioè predisposto per Mathlib:

```
%cd /content
!lake new Project math
```

Lake, tra le altre cose, ha ora creato un file `.toml` chiamato `lakefile` e contenente le $dipendenze$, cioè i file che avrà bisgono di scaricre affinché il progetto funzioni correttamente. Possiamo scaricarli entrando nella cartella e dando il comando `update`:

```
%cd /content/Project
!lake update
```

Sorge nuovamente il problema di riproducibilità. Come assicurare che anche la versione di Mathlib installata sia sempre la stessa?

Con `update`, la versione dei file scaricati è stata registrata in un file `.json` chiamato `lake-manifest`. L'unico modo per garantire che la versione installata di Mathlib sia sempre la stessa è scrivere i file `lakefile` e `lake-manifest` manualmente: così, si scaricheranno ogni volta solo i file relativi alle medesime dipendenze, nella medesima versione.

Usiamo quindi il comando `cat` per visualizzare il contenuto di questi due file:

```
!cat /content/Project/lean-toolchain /content/Project/lakefile.toml /content/Project/lake-manifest.json
```

Al fine di poterli ri-scrivere manualmente, copio-incolliamo il loro contenuto assegnandolo a due variabili, rispettivamente `lakefile_toml` e `lake_manifest_json`. La seguente cella creerà tali variabili nel nostro ambiente.

In [ ]:
lakefile_toml = """\
name = "Project"
version = "0.1.0"
keywords = ["math"]
defaultTargets = ["Project"]

[leanOptions]
pp.unicode.fun = true # pretty-prints `fun a ↦ b`
relaxedAutoImplicit = false
weak.linter.mathlibStandardSet = true
maxSynthPendingDepth = 3

[[require]]
name = "mathlib"
scope = "leanprover-community"
rev = "v4.34.1"

[[lean_lib]]
name = "Project"
"""

lake_manifest_json = """\
{"version": "1.2.0",
 "packagesDir": ".lake/packages",
 "packages":
 [{"url": "https://github.com/leanprover-community/mathlib4",
   "type": "git",
   "subDir": null,
   "scope": "leanprover-community",
   "rev": "d13f23b723b8a846827a245b89c10fc7d3f11612",
   "name": "mathlib",
   "manifestFile": "lake-manifest.json",
   "inputRev": "v4.34.1",
   "inherited": false,
   "configFile": "lakefile.lean"},
  {"url": "https://github.com/leanprover-community/plausible",
   "type": "git",
   "subDir": null,
   "scope": "leanprover-community",
   "rev": "118aa17ee84656b8bd727fef7c458ee8c833385c",
   "name": "plausible",
   "manifestFile": "lake-manifest.json",
   "inputRev": "main",
   "inherited": true,
   "configFile": "lakefile.toml"},
  {"url": "https://github.com/leanprover-community/LeanSearchClient",
   "type": "git",
   "subDir": null,
   "scope": "leanprover-community",
   "rev": "ddf04cf3949fa556442341e87d47f9f6e6074707",
   "name": "LeanSearchClient",
   "manifestFile": "lake-manifest.json",
   "inputRev": "main",
   "inherited": true,
   "configFile": "lakefile.toml"},
  {"url": "https://github.com/leanprover-community/import-graph",
   "type": "git",
   "subDir": null,
   "scope": "leanprover-community",
   "rev": "e928b72544873815af278d38681b31c0293588e3",
   "name": "importGraph",
   "manifestFile": "lake-manifest.json",
   "inputRev": "main",
   "inherited": true,
   "configFile": "lakefile.toml"},
  {"url": "https://github.com/leanprover-community/ProofWidgets4",
   "type": "git",
   "subDir": null,
   "scope": "leanprover-community",
   "rev": "106ff4fafc74ef4ac99d81dbf3ab399118f497a5",
   "name": "proofwidgets",
   "manifestFile": "lake-manifest.json",
   "inputRev": "main",
   "inherited": true,
   "configFile": "lakefile.lean"},
  {"url": "https://github.com/leanprover-community/aesop",
   "type": "git",
   "subDir": null,
   "scope": "leanprover-community",
   "rev": "355695d523e41d0554926416cba2a2b3544fbbc9",
   "name": "aesop",
   "manifestFile": "lake-manifest.json",
   "inputRev": "master",
   "inherited": true,
   "configFile": "lakefile.toml"},
  {"url": "https://github.com/leanprover-community/quote4",
   "type": "git",
   "subDir": null,
   "scope": "leanprover-community",
   "rev": "6a489d9af5d0c47e5b259e2e8bcdfc1811b5a259",
   "name": "Qq",
   "manifestFile": "lake-manifest.json",
   "inputRev": "master",
   "inherited": true,
   "configFile": "lakefile.toml"},
  {"url": "https://github.com/leanprover-community/batteries",
   "type": "git",
   "subDir": null,
   "scope": "leanprover-community",
   "rev": "f2effa3d803fda822b1f97b806c47cf2adfbcbc2",
   "name": "batteries",
   "manifestFile": "lake-manifest.json",
   "inputRev": "main",
   "inherited": true,
   "configFile": "lakefile.toml"},
  {"url": "https://github.com/leanprover/lean4-cli",
   "type": "git",
   "subDir": null,
   "scope": "leanprover",
   "rev": "e92c9f15fdfacc8536f31cfb3b7ad26c3c8cd204",
   "name": "Cli",
   "manifestFile": "lake-manifest.json",
   "inputRev": "v4.34.0",
   "inherited": true,
   "configFile": "lakefile.toml"}],
 "name": "Project",
 "lakeDir": ".lake",
 "fixedToolchain": false}
"""

Quindi, andiamo a usare `open` e `write` per scrivere manualmente i due file. Questo rende superfluo il comando `math`: un progetto generico sarà sufficiente. Poiché il progetto potrebbe creare un file `lean-toolchain` che specifica una versione di Lean diversa da quella che abbiamo installato (la $4.34.1$), andiamo per sicurezza a riscriverlo, specificando la versione in uso. Poiché non vogliamo riscrivere questi file ad ogni esecuzione della cella, poniamo il tutto sotto un `if` che verrà eseguito solo se il file `lake-manifest` (l'ultimo della lista) non è ancora stato creato:

```
if not os.path.isfile("/content/Project/lake-manifest.json"):
  %cd /content
  !lake new Project
  with open("/content/Project/lean-toolchain", "w") as f: f.write("leanprover/lean4:v4.34.1\n")
  with open("/content/Project/lakefile.toml", "w") as f: f.write(lakefile_toml)
  with open("/content/Project/lake-manifest.json", "w") as f: f.write(lake_manifest_json)
```

Dobbiamo ora gestire un caso problematico: se il codice si interrompe per errore prima della fine, il `Project` potrebbe già esistere, benché incompleto. Per sicurezza, possiamo usare la funzione `rmtree` di `shutil` per cancellare la cartella nel caso questa esista già, specificando l'argomento `ignore_errors=True` onde evitare errori nel caso invece non esistesse:

```
if not os.path.isfile("/content/Project/lake-manifest.json"):
  shutil.rmtree("/content/Project", ignore_errors=True)
  %cd /content
  !lake new Project
  with open("/content/Project/lean-toolchain", "w") as f: f.write("leanprover/lean4:v4.34.1\n")
  with open("/content/Project/lakefile.toml", "w") as f: f.write(lakefile_toml)
  with open("/content/Project/lake-manifest.json", "w") as f: f.write(lake_manifest_json)
```

Benché non sia più necessario eseguire `update` nella cartella del progetto, rimane necessario *compilare* i file. Infatti, i risultati matematici scaricati da Mathlib sono sorgenti con estensione `.lean`. Affinché tali risultati siano importabili da altre dimostrazioni, è necessario trasformarli in file `.olean` utilizzabili. Poiché questa operazione può essere molto lunga, usiamo il comando `exe cache get`, che esegue il programma `cache` per scaricare delle copie di file già compilati:

```
%cd /content/Project
!lake exe cache get
```

Ora verifichiamo che l'installazione sia andata a buon fine. Creiamo quindi un file `.lean` che includa il comando `import Mathlib`:

```
with open("/content/Project/test.lean", "w") as f: f.write("import Mathlib\n#eval 1 + 1\n")
```

Andiamo quindi a usare la funzione `run` di `subprocess`. Come primo argomento, inseriamo una lista `"lake", "env", "lean", "/content/Project/test.lean"`. Con `env lean` si chiede al programma `lake` (per il quale, essendo stato aggiunto al `PATH`, non è necessario indicare il percorso) di preparare le variabili d'ambiente e di eseguire lean sul file `.lean` indicato. Segue la richiesta di catturare l'output come testo:

```
result = subprocess.run(["lake", "env", "lean", "/content/Project/test.lean"], capture_output=True, text=True)
```

La stampa del risultato è analoga a quella dell'installazione del core. Di seguito è possibile vedere il codice completo ed eseguirlo. Si consideri che il download dei file tramite `cache` richiederà circa un quarto d'ora (senza `cache`, la loro compilazione richiederebbe diverse ore).

In [ ]:
if not os.path.isfile("/content/Project/lake-manifest.json"):
    shutil.rmtree("/content/Project", ignore_errors=True)
    %cd /content
    !lake new Project
    with open("/content/Project/lean-toolchain", "w") as f: f.write("leanprover/lean4:v4.34.1\n")
    with open("/content/Project/lakefile.toml", "w") as f: f.write(lakefile_toml)
    with open("/content/Project/lake-manifest.json", "w") as f: f.write(lake_manifest_json)

%cd /content/Project
!lake exe cache get

with open("/content/Project/test.lean", "w") as f: f.write("import Mathlib\n#eval 1 + 1\n")

result = subprocess.run(["lake", "env", "lean", "/content/Project/test.lean"], capture_output=True, text=True)

if result.returncode == 0:
    print("✅ Mathlib has been installed successfully")
else:
    print("❌ Something went wrong, try again")
    print(result.stderr)

/content/Project
info: mathlib: cloning https://github.com/leanprover-community/mathlib4
info: mathlib: checking out revision 'd13f23b723b8a846827a245b89c10fc7d3f11612'
info: plausible: cloning https://github.com/leanprover-community/plausible
info: plausible: checking out revision '118aa17ee84656b8bd727fef7c458ee8c833385c'
info: LeanSearchClient: cloning https://github.com/leanprover-community/LeanSearchClient
info: LeanSearchClient: checking out revision 'ddf04cf3949fa556442341e87d47f9f6e6074707'
info: importGraph: cloning https://github.com/leanprover-community/import-graph
info: importGraph: checking out revision 'e928b72544873815af278d38681b31c0293588e3'
info: proofwidgets: cloning https://github.com/leanprover-community/ProofWidgets4
info: proofwidgets: checking out revision '106ff4fafc74ef4ac99d81dbf3ab399118f497a5'
info: aesop: cloning https://github.com/leanprover-community/aesop
info: aesop: checking out revision '355695d523e41d0554926416cba2a2b3544fbbc9'
info: Qq: cloning ht

###4.2 Wrapper

Con Vampire e Z3 abbiamo gestito il sistema logico, le premesse, e la conclusione da derivare come argomenti diversi della funzione wrapper. Con Lean tale distinzione non è utile: infatti il kernel controlla la correttezza di ogni parte del codice, incluse le dichiarazioni. Ogni fase, quindi, è trattata come un file da controllare. La possibilità di combinare un sistema logico con insiemi di premesse diversi va gestita tramite le importazioni dei file `.olean` all'interno di Lean stesso.

Dunque, definiamo una funzione `check_lean` che accetta due argomenti: il codice della prova `lean_code` e il nome `name` da dare ai file:

```
def check_lean(lean_code, name):
```

Creiamo quindi le variabili dei nomi dei file. Dobbiamo creare sia il file `.lean` che contiene il codice della prova, sia il file `.olean` che conterrà il codice compilato da Lean eventualmente importabile. Proprio per consentire l'importazione, creeremo un file separato per ogni codice (mentre Vampire andava a sovrascrivere sempre lo stesso file `theorem.p`). Usiamo una *stringa f* per assegnare automaticamente il nome scelto `name` per i file. Usiamo quindi `open` e `write` per scrivere il codice `lean_code` nel file `.lean`:

```
lean_file = f"{name}.lean"
olean_file = f"{name}.olean"

with open(lean_file, "w") as f: f.write(lean_code)
```

Come negli altri codici, catturiamo con la variabile `result` l'output della funzione `run`.

Come primo argomento, inseriamo una lista di comandi per il programma `lake`. A differenza di quello visto durante l'installazione di Mathlib, aggiungiamo `"-o", olean_file`, che indica il file `.olean` come quello in cui si andrà a scrivere l'output della compilazione di `lean_file`:

```
result = subprocess.run(["lake", "env", "lean", "-o", olean_file, lean_file])
```

Come secondo argomento andiamo a definire le variabili d'ambiente in `env`: importiamo tutte le variabili di Python contenute in `os.environ` con il comando `**`, e poi aggiungiamo la voce `"LEAN_PATH": "."`, che pone la cartella corrente come il percorso dove cercare i moduli importabili (come gli stessi file `.olean` lì salvati). Come sempre, inseriamo i comandi per catturare l'output di `run` come testo:

```
result = subprocess.run(["lake", "env", "lean", "-o", olean_file, lean_file], env={**os.environ, "LEAN_PATH": "."}, capture_output=True, text=True)
```

La stampa dell'output è gestita in modo simile agli altri wrapper. Il codice completo può essere eseguito nella prossima cella.

In [ ]:
def check_lean(lean_code, name):
  lean_file = f"{name}.lean"
  olean_file = f"{name}.olean"

  with open(lean_file, "w") as f: f.write(lean_code)

  result = subprocess.run(["lake", "env", "lean", "-o", olean_file, lean_file], env={**os.environ, "LEAN_PATH": "."}, capture_output=True, text=True)

  if result.returncode == 0:
    print("✅ Kernel check passed")
    if result.stdout.strip():
        print("\n" + result.stdout)
  else:
    print("❌ Error")
    print("STDOUT:", result.stdout)
    print("STDERR:", result.stderr)

print("✅ The wrapper has been loaded")

✅ The wrapper has been loaded


###4.3 Provare un teorema in Lean

Nella teoria dei tipi di Lean, una dimostrazione di una conclusione $C$ è un termine avente tipo $C$. Questo termine è indicato dal proof assistant come *obiettivo* da costruire.

Lean, come molti proof assistants, lavora tramite *tattiche*: piccoli algoritmi che trasformano, in vari modi, l'obiettivo. L'idea è quella di applicare tattiche finché l'obiettivo non rimane vuoto: ciò significa che si è prodotto un termine del tipo della conclusione, cioè una dimostrazione.

Lean è uno strumento molto complesso da utilizzare: esistono infatti decine di tattiche da imparare, non sempre facili da comprendere, nonché decine di moduli importabili ognuno con le sue particolarità. Di seguito ci limiteremo quindi ad un semplice esempio numerico che permette di cogliere didatticamente gli aspetti fondamentali di Lean.

Poniamo un sistema di tre equazioni:

$
\begin{cases}
a = 2 \\
b = a + 1 \\
c = b + a
\end{cases}
$

La conclusione che vogliamo dimostrare è

$c = 5$

Impostiamo quindi il teorema che dimostrerà che $c = 5$. La stringa che dichiara il teorema ha questa struttura:

$\color{red}{\text{theorem}} \color{#56B4E9}{\text{ nome_teorema (premessa)}} \color{red}{:} \color{#56B4E9}{\text{proposizione_da_dimostrare}} \color{red}{:=\ by}$

In particolare, come premesse indichiamo le tre formule del sistema, chiamandole rispettivamente `h_1`, `h_2` e `h_3`:

```
theorem Example (h_1 : a = 2) (h_2 : b = a + 1) (h_3 : c = b + a) : c = 5 := by
```

Tutta la dimostrazione andrà indentata sotto tale riga, la quale pone come obiettivo da dimostrare

```
goal: c = 5
```

Iniziamo usando solo la tattica `rewrite`, la quale riscrive l'obiettivo. Essa si dichiara come

$\color{red}{\text{rewrite [}} \color{#56B4E9}{\text{nome_teorema}} \color{red}{]}$

dove ci si riferisce solitamente ad un teorema della forma $a = b$. La tattica riscrive tutte le istanze di $a$ nell'obiettivo come $b$ (o viceversa). Quindi scrivendo

```
rewrite [h_3]
```

l'obiettivo diventa

```
goal: b + a = 5
```

cioè, l'istanze di `c` è stata riscritta secondo la sua definizione in `h_3`. Possiamo quindi continuare con:

```
rewrite [h_2]
```

per rendere l'obiettivo:

```
goal: a + 1 + a = 5
```

Infine, con

```
rewrite [h_1]
```

si ottiene

```
goal: 2 + 1 + 2 = 5
```

A questo punto basta applicare la tattica `rfl`, che chiude la dimostrazione quando le due parti dell'uguaglianza sono identiche.

Per controllare la correttezza della derivazione, assegniamo il nostro codice alla variabile `example` e la usiamo come argomento del wrapper `check_lean`.

In [ ]:
example = r"""
theorem Example (h_1 : a = 2) (h_2 : b = a + 1) (h_3 : c = b + a) : c = 5 := by
  rewrite [h_3]
  rewrite [h_2]
  rewrite [h_1]
  rfl
"""

check_lean(example, "example")

✅ Kernel check passed


###4.4 Un esercizio

In Lean, per dimostrare che $2+2=4$ è sufficiente la tattica $rlf$. Infatti, il sistema che rappresenta i numeri naturali in Lean, `Nat`, è in grado di calcolare direttamente il risultato di $2+2$, di modo che le due parti dell'uguaglianza risultano già uguali.

A fini ditattici, formalizziamo una teoria dei numeri naturali più debole, `MyNat`. In tale sistema $2+2$ non viene automaticamente calcolato come identico a $4$, ma le due parti dell'uguale vanno trasformate finché non sono del tutto identiche.

Definiamo quindi dei lemmi utilizzabili:

- `add_zero: a + a = 0`
- `add_succ: a + succ b = succ (a + b)`
- `two_eq_succ_one: 2 = succ 1`
- `three_eq_succ_two: 3 = succ 2`
- `four_eq_succ_three: 4 = succ 3`
- `succ_eq_add_one: succ n = n + 1`

Il seguente esercizio richiede di dimostrare che $2+2=4$ usando solo le tattiche `rlf` e `rewrite` con i lemmi sopra elencati.

In [ ]:
EXERCISE = r"""
-- Definiamo, a scopo didattico, una teoria dei numeri naturali più debole di quella nativa di Lean
inductive MyNat where
  | zero : MyNat
  | succ (n : MyNat) : MyNat
namespace MyNat
@[irreducible] def add : MyNat → MyNat → MyNat
  | a, zero => a
  | a, succ b => succ (add a b)
instance : Add MyNat := ⟨add⟩
@[irreducible] def ofNat' : Nat → MyNat
  | 0 => zero
  | n + 1 => succ (ofNat' n)
instance (n : Nat) : OfNat MyNat n := ⟨ofNat' n⟩

-- Lemmi utilizzabili
theorem add_zero (a : MyNat) : a + 0 = a := by with_unfolding_all rfl
theorem add_succ (a b : MyNat) : a + succ b = succ (a + b) := by with_unfolding_all rfl
theorem two_eq_succ_one : (2 : MyNat) = succ 1 := by with_unfolding_all rfl
theorem three_eq_succ_two : (3 : MyNat) = succ 2 := by with_unfolding_all rfl
theorem four_eq_succ_three : (4 : MyNat) = succ 3 := by with_unfolding_all rfl
theorem succ_eq_add_one (n : MyNat) : succ n = n + 1 := by with_unfolding_all rfl

-- Dimostrare il seguente teorema usando solo le tattiche "rfl" e "rewrite" e i lemmi sopra indicati.
theorem two_plus_two_eq_four : (2 : MyNat) + (2 : MyNat) = (4 : MyNat) := by



"""

check_lean(EXERCISE, "Exercise")

❌ Error
STDOUT: Exercise.lean:25:74: error: unsolved goals
⊢ 2 + 2 = 4

STDERR: 


Questo esercizio ne formalizza uno disponibile in https://adam.math.hhu.de/. Questo sito ne include molti altri per imparare a usare Lean da zero divertendosi.

##Contatti

mail: alessio@hermesambiente.it

Ringrazio sin d'ora chi vorrà consigliare miglioramenti per rendere questo notebook più chiaro e più utile a chi sta imparando ad usare questi strumenti.

##Licenza

The MIT License (MIT)

Copyright (c) 2026 Alessio Montagner

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.